<a href="https://colab.research.google.com/github/Ramdharshan2007/DAA-Lab-Experiment/blob/main/4C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq

def dijkstra_with_hops(graph, source):
    """
    Modified Dijkstra's Algorithm that tracks hop count.
    If two paths have the same total weight, it prefers the one with fewer hops.
    """
    n = len(graph)

    # Store both distance and hop count. Initialize to infinity.
    dist = [float('inf')] * n
    hops = [float('inf')] * n
    prev = [None] * n

    dist[source] = 0
    hops[source] = 0

    # Priority queue stores tuples of (distance, hops, vertex)
    # Python's heapq automatically sorts by distance first, then hops!
    pq = [(0, 0, source)]

    while pq:
        d, h, u = heapq.heappop(pq)

        # Optimization: If we pulled a state from the heap that is worse
        # than our currently known best distance/hops for vertex u, skip it.
        if d > dist[u] or (d == dist[u] and h > hops[u]):
            continue

        for v, weight in graph.get(u, []):
            new_dist = d + weight
            new_hops = h + 1

            # Relaxation condition:
            # 1. We found a strictly shorter distance OR
            # 2. The distance is the same, but we found a path with fewer hops
            if new_dist < dist[v] or (new_dist == dist[v] and new_hops < hops[v]):
                dist[v] = new_dist
                hops[v] = new_hops
                prev[v] = u
                heapq.heappush(pq, (new_dist, new_hops, v))

    return dist, hops, prev

def reconstruct_path(prev, source, target):
    """Reconstructs the path from source to target."""
    path = []
    current = target
    while current is not None:
        path.append(current)
        current = prev[current]
    path.reverse()

    if path and path[0] == source:
        return path
    return []

# --- Main Execution ---
if __name__ == '__main__':
    # 6-Vertex Graph designed to test the hop-count tie-breaker
    # Format: {u: [(v, weight), ...]}
    graph = {
        0: [(1, 5), (3, 10), (2, 2)],
        1: [(3, 5)],                   # Path 0->1->3 costs 10, 2 hops. 0->3 costs 10, 1 hop.
        2: [(4, 3)],
        3: [(5, 2)],
        4: [(5, 4)],
        5: [(0, 12)]                   # Path 0->5 directly costs 12, 1 hop. Path 0->3->5 costs 12, 2 hops.
    }

    # Ensure all vertices exist in the graph dictionary
    for i in range(6):
        if i not in graph:
            graph[i] = []

    source_vertex = 0
    total_vertices = len(graph)

    print(f"--- Modified Dijkstra's Algorithm (Distance & Hops) ---")
    print(f"Source Vertex: {source_vertex}\n")

    distances, hop_counts, predecessors = dijkstra_with_hops(graph, source_vertex)

    print(f"{'Destination':<12} | {'Distance':<10} | {'Hops':<6} | {'Optimal Path'}")
    print("-" * 65)

    for v in range(total_vertices):
        if distances[v] == float('inf'):
            print(f"{v:<12} | {'Unreachable':<10} | {'-':<6} | None")
        else:
            path = reconstruct_path(predecessors, source_vertex, v)
            path_str = " -> ".join(map(str, path))
            print(f"{v:<12} | {distances[v]:<10} | {hop_counts[v]:<6} | {path_str}")


--- Modified Dijkstra's Algorithm (Distance & Hops) ---
Source Vertex: 0

Destination  | Distance   | Hops   | Optimal Path
-----------------------------------------------------------------
0            | 0          | 0      | 0
1            | 5          | 1      | 0 -> 1
2            | 2          | 1      | 0 -> 2
3            | 10         | 1      | 0 -> 3
4            | 5          | 2      | 0 -> 2 -> 4
5            | 9          | 3      | 0 -> 2 -> 4 -> 5
